In [3]:
import io
import zipfile
import requests
import frontmatter
import pydantic_ai
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
url = 'https://codeload.github.com/debauchee/barrier-wiki/faq/zip/refs/heads/main'
resp = requests.get(url)

In [16]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/master'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [17]:
barrier_docs = read_repo_data('debauchee', 'barrier-wiki')

In [5]:
print(f"barrier docs: {len(barrier_docs)}")

barrier docs: 10


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    barrier_chunks.extend(chunks)

In [8]:
print(type(barrier_chunks[0]))
print(barrier_chunks[0].keys())

<class 'dict'>
dict_keys(['start', 'chunk', 'filename'])


In [9]:
from pprint import pprint
pprint(barrier_chunks[0]['chunk'][:200])

('## Adding Barrier to the Windows Firewall\n'
 '\n'
 'Screenshots captured by [l0o0](https://github.com/l0o0)\n'
 '\n'
 '![search windows '
 'defender](https://user-images.githubusercontent.com/2743744/49057950-ca40bd00-f23c-')


In [10]:
import re
text = barrier_docs[6]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [19]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    #For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1] # "##" + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections
    

In [20]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        barrier_chunks.append(section_doc)

In [21]:
from minsearch import Index

barrier_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

barrier_index.fit(barrier_chunks)

In [14]:
print(barrier_chunks[0].keys())

dict_keys(['filename', 'section'])


In [60]:
barrier_faq = read_repo_data('debauchee', 'barrier-wiki')

In [16]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
record = de_barrier_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

NameError: name 'de_barrier_faq' is not defined

In [18]:
query = 'How do I install barrier on Mac OS?'
v_query = embedding_model.encode(query)

In [19]:
query = 'How do I install barrier on Mac OS?'
results = index.search(query)

NameError: name 'index' is not defined

In [21]:
v_doc = embedding_model.encode(text)

similarity = v_query.dot(v_doc)

In [22]:
from tqdm.auto import tqdm
from minsearch import VectorSearch
import numpy as np

barrier_embeddings = []

for d in tqdm(barrier_chunks):
    v = embedding_model.encode(d['section'])
    barrier_embeddings.append(v)

barrier_embeddings = np.array(barrier_embeddings)

np.save("barrier_embeddings.npy", barrier_embeddings)

barrier_vindex = VectorSearch()
barrier_vindex.fit(barrier_embeddings, barrier_chunks)

  0%|          | 0/37 [00:00<?, ?it/s]

In [22]:
def text_search(query):
    return barrier_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return barrier_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and dedupe results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results

In [10]:
%pprint
pprint(text_search('How do I install barrier in Mac OS?'))

Pretty printing has been turned ON


NameError: name 'pprint' is not defined

In [ ]:
pprint(vector_search('How do I install barrier in Mac OS?'))

In [ ]:
pprint(hybrid_search('how do I install barrier in Linux?'))

In [25]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the Barrier docs.

    Args:
        query(str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned from the Barrier docs.
    """
    return barrier_index.search(query, num_results=5)

In [29]:
system_prompt="""
You are a helpful agent that answers questions about Barrier using context from the Barrier documentation.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

If the search doesn't return relevant results, let the user know and provide general guidance.

"""

In [32]:
from pydantic_ai import Agent

agent = Agent(
    name="barrier_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai-chat:gpt-4o-mini'
)

In [33]:
question = "Can I run Barrier on my Linux machine together with my Windows machine?"

result = await agent.run(user_prompt=question)

In [34]:
pprint(result)

AgentRunResult(output='Yes, you can run Barrier on both your Linux and Windows '
                      'machines. Barrier is designed to work across different '
                      'operating systems, allowing you to share your mouse and '
                      'keyboard between them seamlessly. \n'
                      '\n'
                      'Make sure to install Barrier on both systems; you can '
                      'follow the installation steps specific to each OS. For '
                      "Linux, you'll need to build it from source or install "
                      'it using your package manager, while for Windows, you '
                      'might want to use the installer or build it from source '
                      'as well. \n'
                      '\n'
                      'Once both systems have Barrier running, you can '
                      'configure them to work together. If you need more '
                      'detailed instructions on installation 

In [35]:
question = "Do I need to install bonjour to use Barrier?"
result = await agent.run(user_prompt=question)
pprint(result)

AgentRunResult(output='The search did not yield specific information regarding '
                      'the need to install Bonjour to use Barrier. However, '
                      'generally, Bonjour is not a mandatory requirement for '
                      'Barrier to function. \n'
                      '\n'
                      "If you're experiencing issues related to network "
                      'discovery or connections, installing Bonjour might help '
                      'in certain scenarios, especially when dealing with '
                      'devices on a local network. But for standard operation, '
                      'you should be able to run Barrier without it.\n'
                      '\n'
                      'If you have particular issues you are trying to '
                      'resolve, please provide more details, and I can assist '
                      'you further!')


## Day 5: Evaluation

In [63]:
from typing import List, Any
from pydantic_ai import Agent

def text_search(query: str) -> List[Any]:
    """
        Perform a text-based search on the Barrier documentation.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the Barrier documentation.
    """
    return barrier_index.search(query, num_results=5)

system_prompt = """
You are a helpful assistant for a KVM tool. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.  

When citing the reference, replace "https://github.com/debauchee/barrier/blob/master/barrier-wiki-master" by the full path to the GitHub repository: "https://github.com/debauchee/barrier/wiki/"

Format: [LINK TITLE](FULL_GITHUB_LINK)

Leave a space after the link before any other characters to ensure that the link can be used/clickable.

If the search doesn't return relevant results, let the user know and provide general guidance.

""".strip()

from pydantic_ai import Agent

agent = Agent(
    name="barrier_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai-chat:gpt-4o-mini'
)

# Another version of the agent, v2
agent = Agent(
    name="barrier_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model='openai-chat:gpt-4o-mini'
)

In [64]:
question = "Can I run Barrier across Linux and Windows?"
result = await agent.run(user_prompt=question)

In [65]:
print(result)

AgentRunResult(output='Yes, you can run Barrier across both Linux and Windows. Barrier is designed to allow seamless mouse and keyboard control between multiple machines, including both Linux and Windows operating systems.\n\nFor installation on **Linux**, you can build from source or download binaries suitable for various distributions like Ubuntu, Fedora, and CentOS. The process generally involves installing necessary dependencies and using tools like CMake.\n\nOn **Windows**, you will also need to build from source using tools such as Visual Studio and CMake, or you can opt for pre-built binaries. The steps involve setting up your development environment, including installing Qt and Bonjour SDK.\n\nFor specific details regarding the installation process for each operating system, you can refer to the following links:\n\n- [Building Barrier on Linux](https://github.com/debauchee/barrier/wiki/Building-on-Linux)\n- [Building Barrier on Windows](https://github.com/debauchee/barrier/wiki

Here's what we want to record:
The system prompt that we used
The model
The user query
The tools we use
The responses and the back-and-forth interactions between the LLM and our tools
The final response
To make it simpler, we'll implement a simple logging system ourselves: we will just write logs to json files.
You shouldn't use it in production. In practice, you will want to send these logs to some log collection system, or use specialized LLM evaluation tools like Evidently, LangWatch or Arize Phoenix.
Let's extract all this information from the agent and from the run results:


In [30]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

def log_entry(agent, messages, source="user"):
    tools=[]

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

This code extracts the key information from our agent:
the configuration (name, prompt, model)
available tools
complete message history (user input, tool calls, responses)
We also use ModelMessagesTypeAdapter.dump_python(messages) to convert internal message format into regular Python dictionaries. This makes it easier to save it to JSON and process later.
We also add the source parameter. It tracks where the question came from. We start with "user" but later we'll use AI-generated queries. Sometimes it may be important to tell them apart for analysis.
This code is generic so it will work with any Pydantic AI agent. If you use a different library, you'll need to adjust this code.
Let's write these logs to a folder:


In [61]:
import json
import secrets
from pathlib import Path
from datetime import datetime

LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def normalize_timestamp(ts):
    if isinstance(ts, datetime):
        return ts
    if isinstance(ts, str):
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))
    raise TypeError(f"Unexpected timestamp type: {type(ts)}")

def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1].get('timestamp')
    ts_obj = normalize_timestamp(ts)
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    #print(entry['messages'][-1])
    last_msg = entry['messages'][-1]
    ts = last_msg.get("timestamp")
    if ts is None:
        ts_obj = datetime.now()
    else:
        ts_obj = normalize_timestamp(ts)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return(filepath)

This code:
Creates a logs directory (if not created previously)
Generates unique filenames with timestamp and random hex
Saves complete interaction logs as JSON files
Handles datetime serialization (using the serialized function)
Now we can interact with it and do some vibe checking:


In [74]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

 What is a KVM?


KVM stands for Kernel-based Virtual Machine. It is a virtualization solution for Linux on x86 hardware that allows you to turn your Linux machine into a hypervisor. KVM enables you to run multiple virtual machines (VMs) on a single physical server, each with its own virtual hardware, operating system, and applications.

KVM is part of the Linux kernel and includes various components:

1. **Kernel Module**: The core of KVM is a kernel module that provides the infrastructure for virtual machine management.

2. **QEMU**: KVM is often used with QEMU, a user-space emulator that provides the ability to run various operating systems and architectures.

3. **Filesystem**: KVM manages storage for virtual machines, including disk images and configuration files.

4. **Network**: KVM supports bridging and other networking configurations to enable virtual machines to communicate with each other and the host system.

KVM is widely used in data centers and cloud environments for its efficiency, suppo

WindowsPath('logs/barrier_agent_v2_20260607_215233_41327d.json')

This creates a simple interactive loop where:
User enters a question
Agent processes it and responds
Complete interaction is logged to a file


# LLM as a Judge
You can ask your colleagues to also do a "vibe check", but make sure you record the data. Often collecting 10-20 examples and manually inspecting them is enough to understand how your model is doing.

Don't be afraid of putting manual work into evaluation. Manual evaluation will help you understand edge cases, learn what good responses look like and think of evaluation criteria for automated checks later.

For example, I manually inspected the output and noticed that references are missing. So we will later add it as one of the checks.
So, in our case, we can have the following checks:
- Does the agent follow the instructions?
- Given the question, does the answer make sense?
- Does it include references?
- Did the agent use the available tools?

We don't have to evaluate this manually. Instead, we can delegate this to AI. This technique is called "LLM as a Judge".

The idea is simple: we use one LLM to evaluate the outputs of another LLM. This works because LLMs are good at following detailed evaluation criteria.

Our system prompt for the judge (we'll call it "evaluation agent" because it sounds cooler) can look like that:

In [66]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

Since we expect a very well defined structure of the response, we can use structured output.

We can define a Pydantic class with the expected response structure, and the LLM will produce output that matches this schema exactly.

This is how we do it:


In [67]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

This code defines the structure we expect from our evaluation:
- Each check has a name, justification, and pass/fail result
- The overall evaluation includes a list of checks and a summary

Note that justification comes before check_pass. This makes the LLM reason about the answer before giving the final judgment, which typically leads to better evaluation quality.

With Pydantic AI in order to make the output follow the specified class, we use the parameter output_type:

In [69]:
eval_agent = Agent(
    name='eval-agent',
    model='openai-chat:gpt-5-nano',
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

Usually it's a good idea to evaluate the results of one model (in our case, "gpt-4o-mini") with another model (e.g. "gpt-5-nano"). A different model can catch mistakes, reduce self-bias, and give a second opinion. This makes evaluations more reliable.

We have the instructions, and we have the agent. In order to run the agent, it needs input. We'll start with a template:


In [70]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

We use XML markup because it's easier and more clear for LLMs to understand the input. XML tags help the model see the structure and boundaries of different sections in the prompt.

Let's fill it in. First, define a helper function for loading JSON log files:


In [72]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

We also add the filename in the result - it'll help us with tracking later.

Now let's use it:


In [77]:
log_record = load_log_file('./logs/barrier_agent_v2_20260607_215233_41327d.json')

instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)

In [78]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
    print(check)

The answer is correct and clear but did not follow the instruction to search the course materials or include citations. A revised answer should include references to course materials and formatted citations if available.
check_name='instructions_follow' justification="The answer did not reference any course materials or perform a search as requested in the user's instructions." check_pass=False
check_name='instructions_avoid' justification='No prohibited actions were observed in the answer.' check_pass=True
check_name='answer_relevant' justification='The content explains what KVM is and its role in virtualization.' check_pass=True
check_name='answer_clear' justification='The explanation is structured and easy to understand.' check_pass=True
check_name='answer_citations' justification='No formal references or citations were provided as required by the instruction to cite sources.' check_pass=False
check_name='completeness' justification='Covers core aspects (definition, kernel role, typ

This code:
- Loads a saved interaction log
- Extracts the key components (instructions, question, answer, full log)
- Formats them into the evaluation prompt
- Runs the evaluation agent
- Prints the results
  
Note that we're putting the entire conversation log into the prompt, which is not really necessary. We can reduce it to make it less verbose.
For example, like that:



In [79]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']

            if kind == 'user_prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']

            parts.append(part)

        message = {
            'kind': m['kind'],
            'parts': parts
        }

        log_simplified.append(message)
    return log_simplified

We make it simpler:

- remove timestamps and IDs that aren't needed for evaluation
- replace actual search results with a placeholder
- keep only the essential structure

This is helpful because it reduces the number of tokens we send to the evaluation model, which lowers the costs and speeds up evaluation.
Let's put everything together:


In [80]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output

log_record = load_log_file('./logs/barrier_agent_v2_20260607_215233_41327d.json')
eval1 = await evaluate_log_record(eval_agent, log_record)

We know how to log our data and how to run evals on our logs.

Great. But how do we get more data to get a better understanding of the performance of our model?

# Data Generation

We can ask AI to help. What if we used it for generating more questions? Let's do that.

We can sample some records from our database. Then for each record, ask an LLM to generate a question based on the record. We use this question as input to our agent and log the answers.

Let’s start by defining the question generator:


In [81]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a piece of open source software.

Based on the provided documentation, generate realistic questions that curious users or even users just trying to get their systems working might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general user questions

Generate one question for each record.

""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='openai-chat:gpt-4o-mini',
    output_type=QuestionsList
)

We will send it a bunch of records, and it will generate a question from each of them.

Note: we use a simple way of generating questions. We can use a more complex approach where we also track the source (filename) of the question. If we do it, we can later check if this file was retrieved and cited in the answer. But we won't do it today to make things simpler.

Now let's sample 10 records from our dataset using Python's built-in random.sample function:


In [82]:
import random

sample = random.sample(barrier_faq, 10)#
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions


Now we simply iterate over each of the question, ask our agent and log the results:


In [83]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/16 [00:00<?, ?it/s]

What are the steps to install the KDE runtime and SDK for Barrier using Flatpak?
To install the KDE runtime and SDK for Barrier using Flatpak, follow these steps:

1. **Add the Flathub repository** (if it isn't already added):
   ```shell
   flatpak --user remote-add --if-not-exists flathub https://flathub.org/repo/flathub.flatpakrepo
   ```

2. **Install the KDE platform**:
   Replace `x86_64` with your architecture (such as `i386`, `arm`, `aarch64`, etc.) and `5.12` with the current version you need:
   ```shell
   flatpak --user install flathub org.kde.Platform/x86_64/5.12
   ```

3. **Install the KDE SDK**:
   Again, replace `x86_64` and `5.12` as needed:
   ```shell
   flatpak --user install flathub org.kde.Sdk/x86_64/5.12
   ```

Make sure to check the version of the runtimes and SDK used in the relevant [Barrier manifest JSON file](https://github.com/AdrianKoshka/barrier-manifest/blob/master/com.github.debauchee.barrier.json#L3-L5) for accuracy.

For more detailed instructions, 

We can repeat it multiple times until we have enough data. Around 100 should be good for a start, but today we can just continue with the 10 log records we already generated.

Using AI for generating test data is quite powerful. It can help us get data faster and sometimes cover edge cases we won't think about.

There are limitations too:
- AI-generated questions might not reflect real user behavior
- It may miss important edge cases that only real users encounter
- They may not capture the full complexity of real user queries

The logs are ready, so we can run evaluation on them with our evaluation agent.

First, collect all the AI-generated logs for the v2 agent:


In [90]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'barrier_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

And evaluate them:

In [91]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

  0%|          | 0/16 [00:00<?, ?it/s]

This code:

- Loops through each AI-generated log
- Runs our evaluation agent on it
- Stores both the original log and evaluation result

There are ways to speed this up, but we won't cover them in detail here. For example, you can try this:
- Don't ask for justification - this makes evaluation faster but slightly lower quality
- Parallelize execution - you can ask ChatGPT how to do this with async/await

The results are collected, but we need to display them and also calculate some statistics. The best tool for doing this is Pandas. We already should have it because minsearch depends on it. 


Our data is not ready to be converted to a Pandas DataFrame. We first need to transform it a little. Let’s do it:

In [93]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

This code:

- Extracts key information from each log (file, question, answer)
- Converts the evaluation checks into a dictionary format

Now each row is a simple key-value dictionary, so we can create a DataFrame:


In [94]:
import pandas as pd

df_evals = pd.DataFrame(rows)


We can look at individual records and see which checks are False.

But it's also useful to look at the overall stats:


In [97]:
print(df_evals)

                                            file  \
0   barrier_agent_v2_20260607_221735_ea8f2d.json   
1   barrier_agent_v2_20260607_221740_eddcd1.json   
2   barrier_agent_v2_20260607_221745_a4e24b.json   
3   barrier_agent_v2_20260607_221748_1ff56b.json   
4   barrier_agent_v2_20260607_221755_90c709.json   
5   barrier_agent_v2_20260607_221803_720db4.json   
6   barrier_agent_v2_20260607_221811_3e007d.json   
7   barrier_agent_v2_20260607_221816_d4afb2.json   
8   barrier_agent_v2_20260607_221823_45a55d.json   
9   barrier_agent_v2_20260607_221828_995e47.json   
10  barrier_agent_v2_20260607_221835_1c9c0d.json   
11  barrier_agent_v2_20260607_221839_8e9d4e.json   
12  barrier_agent_v2_20260607_221844_9d999f.json   
13  barrier_agent_v2_20260607_221849_c4a717.json   
14  barrier_agent_v2_20260607_221853_0fa2c8.json   
15  barrier_agent_v2_20260607_221859_28a8b2.json   

                                             question  \
0   What are the steps to install the KDE runtime ...   
1

In [98]:
for col in df_evals.columns[3:]:
    print(col, type(df_evals[col].iloc[0]), df_evals[col].iloc[0])

instructions_follow <class 'bool'> False
instructions_avoid <class 'bool'> True
answer_relevant <class 'bool'> True
answer_clear <class 'bool'> True
answer_citations <class 'bool'> True
completeness <class 'bool'> True
tool_call_search <class 'bool'> True


In [101]:
for col in df_evals.columns[3:]:
    bad = df_evals[~df_evals[col].apply(lambda x: isinstance(x, bool))]
    if len(bad):
        print(col)
        print(bad[[col]])

instructions_follow
  instructions_follow
5                 NaN
instructions_avoid
  instructions_avoid
5                NaN
answer_relevant
  answer_relevant
5             NaN
answer_clear
  answer_clear
5          NaN
answer_citations
  answer_citations
5              NaN
completeness
   completeness
5           NaN
10          NaN
tool_call_search
   tool_call_search
5               NaN
10              NaN


In [105]:
print(eval_results[5])
print(eval_results[10])

({'agent_name': 'barrier_agent_v2', 'system_prompt': ['You are a helpful assistant for a KVM tool. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provide accurate answers.\n\nAlways include references by citing the filename of the source material you used.  \n\nWhen citing the reference, replace "https://github.com/debauchee/barrier/blob/master/barrier-wiki-master" by the full path to the GitHub repository: "https://github.com/debauchee/barrier/wiki/"\n\nFormat: [LINK TITLE](FULL_GITHUB_LINK)\n\nLeave a space after the link before any other characters to ensure that the link can be used/clickable.\n\nIf the search doesn\'t return relevant results, let the user know and provide general guidance.'], 'provider': 'openai', 'model': 'gpt-4o-mini', 'tools': ['text_search'], 'messages': [{'parts': [{'content': 'Can I cross-compile Barrier for a different architecture, 

In [108]:
# Troubleshooting: turning non-booleans into booleans to avoid type being changed to object

check_cols = df_evals.columns[3:]

df_evals[check_cols] = df_evals[check_cols].astype("boolean")

In [107]:
df_evals.select_dtypes(include='bool').mean()

instructions_follow         0.6
instructions_avoid          1.0
answer_relevant        0.933333
answer_clear           0.933333
answer_citations       0.733333
completeness           0.928571
tool_call_search       0.714286
dtype: Float64

This tells us:

- 60% of responses follow instructions completely
- All responses avoid forbidden actions (good!)
- 93% answers are relevant and clear (great!)
- 73% include proper citations (pretty good!)
- 93% of responses are complete
- 71% of responses use the search tool

For us, the most important check is answer_relevant. This tells us whether the agent actually answers the user's question. If this score was low, it’d mean that our agent is not ready.

We now know how to evaluate our agent. What can we do with it now?

Many things:
- Decide if this quality is good enough for deployment
- Evaluate different chunking approaches and search
- See if changing a prompt leads to any improvements.

The algorithm is simple:
- Collect data for evaluation and keep this dataset fixed
- Run different versions of your agent for this dataset
- Compare key metrics to decide which version is better

Evaluation is a very powerful tool and we should use it when possible. 


In [96]:
# Troubleshooting why the above code hadn't worked on first run (returned empty data frame)

print(len(rows))
print(df_evals.shape)
print(df_evals.columns.tolist())
print(df_evals.dtypes)
print(df_evals.head(2))

16
(16, 10)
['file', 'question', 'answer', 'instructions_follow', 'instructions_avoid', 'answer_relevant', 'answer_clear', 'answer_citations', 'completeness', 'tool_call_search']
file                      str
question                  str
answer                    str
instructions_follow    object
instructions_avoid     object
answer_relevant        object
answer_clear           object
answer_citations       object
completeness           object
tool_call_search       object
dtype: object
                                           file  \
0  barrier_agent_v2_20260607_221735_ea8f2d.json   
1  barrier_agent_v2_20260607_221740_eddcd1.json   

                                            question  \
0  What are the steps to install the KDE runtime ...   
1  Can you explain what Immune Keys are in Barrie...   

                                              answer instructions_follow  \
0  To install the KDE runtime and SDK for Barrier...               False   
1  **Immune Keys** in Barrier ar

# Evaluating functions and tools

Also, we can (and should) evaluate our tools separately from evaluating the agent.

If it's code, we need to cover it with unit and integration tests.

We also have the search function, which we can evaluate using standard information retrieval metrics. For example:
- Precision and Recall: How many relevant results were retrieved vs. how many relevant results were missed
- Hit Rate: Percentage of queries that return at least one relevant result
- MRR (Mean Reciprocal Rank): Reflects the position of the first relevant result in the ranking

This is how we can implement hitrate and MRR calculation in Python:


In [109]:
def evaluate_search_quality(search_function, test_queries):
    results=[]

    for query, expected_docs in test_queries:
        search_results = search_function(query, num_results=5)

        # Calculate hit rate
        relevant_found = any(doc['filename'] in expected_docs for doc in search_results)

        # Calculate MMR
        for i, doc in enumerate(search_results):
            if doc['filename'] in expected_docs:
                mrr = 1 / (i+1)
                break
        else:
            mrr = 0

        results.append({
            'query': query,
            'hit': relevant_found,
            'mrr': mrr
        })
    return results

We won't do it today, but these ideas and the code will be useful when you implement a real agent project with search.

It's useful because it'll helps us make guided decisions about:
- When to use text vs. vector vs. hybrid search
- What are the best parameters for our search

You can ask ChatGPT to learn more about information retrieval evaluation metrics.
